# NAT LinearPool Latent Tests

This notebook is a clean rewrite of the post-training latent-space section from `train-NAT-LinearPool.ipynb`.

It keeps the same test categories for the final NAT LinearPool checkpoint, removes the unrelated refresh workflow, and normalizes the helper logic so the notebook can be run top-to-bottom.

In [ ]:
import importlib.util
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import selfies as sf
import torch
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import DataStructs, Draw, rdFingerprintGenerator
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

cwd = Path.cwd()
if (cwd / 'transformer_vae').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'transformer_vae').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Could not locate the project root from the current working directory.')

TRANSFORMER_DIR = PROJECT_ROOT / 'transformer_vae'
DATA_CSV = PROJECT_ROOT / 'smiles_selfies_full.csv'
MODEL_PY = TRANSFORMER_DIR / 'models' / 'final_nat_linearpool.py'
CKPT_PATH = TRANSFORMER_DIR / 'trained_models' / 'final(H256-L512-linpool).pt'
LATENT_TEST_DIR = PROJECT_ROOT / 'artifacts' / 'latent_test_analysis'
PLOTS_DIR = LATENT_TEST_DIR / 'plots'
TABLES_DIR = LATENT_TEST_DIR / 'tables'
NOTES_DIR = LATENT_TEST_DIR / 'notes'

assert DATA_CSV.exists(), f'Missing dataset: {DATA_CSV}'
assert MODEL_PY.exists(), f'Missing model helper: {MODEL_PY}'
assert CKPT_PATH.exists(), f'Missing checkpoint: {CKPT_PATH}'

for folder in (LATENT_TEST_DIR, PLOTS_DIR, TABLES_DIR, NOTES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('project_root:', PROJECT_ROOT)
print('checkpoint:', CKPT_PATH)
print('latent_test_artifacts:', LATENT_TEST_DIR)


## Setup

This setup mirrors the training notebook: the CSV is re-tokenized into the same vocabulary, the padded arrays are split with the same random seeds, and the final NAT LinearPool checkpoint is loaded through the shared helper module.

In [ ]:
df = pd.read_csv(DATA_CSV)
required_cols = {'smiles', 'selfies'}
assert required_cols.issubset(df.columns), f'CSV must contain {required_cols}'

df['tokens'] = df['selfies'].apply(lambda x: list(sf.split_selfies(x)))
all_tokens = [tok for seq in df['tokens'] for tok in seq]

PAD = '<PAD>'
SOS = '<SOS>'
EOS = '<EOS>'
vocab = [PAD, SOS, EOS] + sorted(set(all_tokens))
vocab_size = len(vocab)

tok2id = {tok: idx for idx, tok in enumerate(vocab)}
id2tok = {idx: tok for tok, idx in tok2id.items()}

PAD_ID = tok2id[PAD]
SOS_ID = tok2id[SOS]
EOS_ID = tok2id[EOS]

def full_molecule_tokens_to_ids(tokens, tok2id):
    return np.array([tok2id[SOS]] + [tok2id[t] for t in tokens] + [tok2id[EOS]], dtype=np.int64)

df['token_ids'] = df['tokens'].apply(lambda toks: full_molecule_tokens_to_ids(toks, tok2id))
df['lenghts'] = df['token_ids'].apply(len)
df['lengths'] = df['lenghts']

sequences = df['token_ids'].tolist()
max_len = max(len(seq) for seq in sequences)
padded_data = np.zeros((len(sequences), max_len), dtype=np.int64)
for i, seq in enumerate(sequences):
    padded_data[i, :len(seq)] = seq

data = padded_data
train_data, temp_data = train_test_split(data, test_size=0.2, random_state=42, shuffle=True)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, shuffle=True)

spec = importlib.util.spec_from_file_location('final_nat_linearpool', MODEL_PY)
final_nat_linearpool = importlib.util.module_from_spec(spec)
spec.loader.exec_module(final_nat_linearpool)

latent_size = final_nat_linearpool.FINAL_LATENT_SIZE

def load_final_model_safe(vocab_size, max_len, ckpt_path, device):
    model = final_nat_linearpool.VaeTransformer(
        vocab_size=vocab_size,
        hidden_size=final_nat_linearpool.FINAL_HIDDEN_SIZE,
        latent_size=final_nat_linearpool.FINAL_LATENT_SIZE,
        max_len=max_len,
        attn_heads=final_nat_linearpool.FINAL_ATTENTION_HEADS,
        num_slots=final_nat_linearpool.FINAL_NUM_SLOTS,
        layers=final_nat_linearpool.FINAL_LAYERS,
    ).to(device)

    load_kwargs = {'map_location': device}
    try:
        ckpt = torch.load(str(ckpt_path), weights_only=True, **load_kwargs)
    except TypeError:
        ckpt = torch.load(str(ckpt_path), **load_kwargs)

    state_dict = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
    model.load_state_dict(state_dict)
    model.eval()
    return model

model = load_final_model_safe(
    vocab_size=vocab_size,
    max_len=max_len,
    ckpt_path=CKPT_PATH,
    device=device,
).to(device)
model.eval()

print(f'Data shapes: Train {train_data.shape}, Val {val_data.shape}, Test {test_data.shape}')
print('vocab_size:', vocab_size)
print('max_len:', max_len)
print('latent_size:', latent_size)


In [ ]:
def normalize_ids(token_ids):
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.detach().cpu().tolist()
    return [int(t) for t in token_ids]

def cleaned_tokens_from_ids(token_ids):
    cleaned = []
    for token_id in normalize_ids(token_ids):
        if token_id == EOS_ID:
            break
        if token_id in (PAD_ID, SOS_ID):
            continue
        tok = id2tok.get(token_id)
        if tok is None or tok in (PAD, SOS):
            continue
        if tok == EOS:
            break
        cleaned.append(tok)
    return cleaned

def ids_to_selfies(token_ids):
    return ''.join(cleaned_tokens_from_ids(token_ids))

def selfies_to_smiles(selfies_str):
    if not selfies_str:
        return None
    try:
        return sf.decoder(selfies_str)
    except Exception:
        return None

def mol_from_smiles(smiles_str):
    if not smiles_str:
        return None
    try:
        return Chem.MolFromSmiles(smiles_str)
    except Exception:
        return None

def canonical_smiles(smiles_str):
    mol = mol_from_smiles(smiles_str)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

def ids_to_smiles(token_ids):
    selfies_str = ids_to_selfies(token_ids)
    return selfies_to_smiles(selfies_str)

def tensorize_rows(rows):
    if isinstance(rows, torch.Tensor):
        return rows.to(device)
    return torch.as_tensor(rows, dtype=torch.long, device=device)

def unpack_decoder_output(decoder_out):
    if isinstance(decoder_out, (tuple, list)):
        logits = decoder_out[0]
        pred_len = decoder_out[1] if len(decoder_out) > 1 else None
    else:
        logits = decoder_out
        pred_len = None
    return logits, pred_len

def logits_to_ids(logits):
    logits = torch.as_tensor(logits)
    if logits.dim() == 3:
        logits = logits[0]
    if logits.dim() == 2:
        return logits.argmax(dim=-1).detach().cpu().tolist()
    if logits.dim() == 1:
        return [int(logits.argmax().item())]
    raise ValueError(f'Unexpected logits rank: {logits.dim()}')

def trim_pred_ids(token_ids, pred_len_value=None):
    ids = normalize_ids(token_ids)
    if pred_len_value is not None:
        pred_len_value = max(1, int(round(float(pred_len_value))))
        ids = ids[:pred_len_value]
    return ids

def decode_latent_batch(z_batch, sos_id=SOS_ID):
    z_batch = torch.as_tensor(z_batch, dtype=torch.float32, device=device)
    if z_batch.dim() == 1:
        z_batch = z_batch.unsqueeze(0)
    with torch.no_grad():
        decoder_out = model.decode(z_batch, mode='eval', sos_id=sos_id)
    logits, pred_len = unpack_decoder_output(decoder_out)
    pred_ids = logits.argmax(dim=-1)
    rows = []
    for i in range(pred_ids.size(0)):
        ids = pred_ids[i].detach().cpu().tolist()
        length_i = None if pred_len is None else float(pred_len[i].item())
        ids = trim_pred_ids(ids, length_i)
        selfies_str = ids_to_selfies(ids)
        smiles_str = selfies_to_smiles(selfies_str)
        canon = canonical_smiles(smiles_str)
        rows.append({
            'pred_ids': ids,
            'selfies': selfies_str,
            'smiles': smiles_str,
            'canonical_smiles': canon,
            'is_valid': int(canon is not None),
        })
    return pd.DataFrame(rows)

def encode_numpy_rows(rows, batch_size=256):
    rows = np.asarray(rows)
    chunks = []
    with torch.no_grad():
        for start in range(0, len(rows), batch_size):
            batch = tensorize_rows(rows[start:start + batch_size])
            mu, _ = model.encode(batch)
            chunks.append(mu.detach().cpu().numpy())
    return np.concatenate(chunks, axis=0)

def smiles_to_ids(smiles, tok2id=tok2id, max_len=max_len):
    selfies = sf.encoder(smiles)
    tokens = list(sf.split_selfies(selfies))
    ids = [tok2id[SOS]] + [tok2id[t] for t in tokens] + [tok2id[EOS]]
    padded = np.zeros(max_len, dtype=np.int64)
    padded[:len(ids)] = ids
    return padded

def encode_smiles(smiles):
    x = tensorize_rows(smiles_to_ids(smiles)).unsqueeze(0)
    with torch.no_grad():
        mu, _ = model.encode(x)
    return mu.squeeze(0)

def count_duplicates(values):
    if not values:
        return [], []
    unique = [values[0]]
    counts = [1]
    for value in values[1:]:
        if value == unique[-1]:
            counts[-1] += 1
        else:
            unique.append(value)
            counts.append(1)
    return unique, counts

def write_note(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text.strip() + '\n', encoding='utf-8')

def save_rdkit_image(image_obj, out_path):
    out_path = Path(out_path)
    if isinstance(image_obj, (bytes, bytearray)):
        out_path.write_bytes(bytes(image_obj))
    elif hasattr(image_obj, 'save'):
        image_obj.save(str(out_path))
    elif hasattr(image_obj, 'data') and isinstance(image_obj.data, (bytes, bytearray)):
        out_path.write_bytes(bytes(image_obj.data))
    else:
        raise TypeError(f'Unsupported image type from RDKit draw helper: {type(image_obj)}')

MORGAN_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def make_morgan_fp(mol, radius=2, nBits=2048):
    if mol is None:
        return None
    if radius == 2 and nBits == 2048:
        return MORGAN_GENERATOR.GetFingerprint(mol)
    return rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nBits).GetFingerprint(mol)


## Reconstruction Spot Checks

This is the same post-training sanity check as in the training notebook: sample a few held-out molecules, run the model in evaluation mode, and compare the reconstructed SELFIES and SMILES.

The prediction indexing is fixed here so the decoded tokens are taken from the model output directly instead of being wrapped incorrectly.

In [ ]:
rows = []
for _ in range(5):
    i = np.random.randint(0, len(test_data))
    x = tensorize_rows(test_data[i:i + 1])
    with torch.no_grad():
        logits, _, _, pred_len = model(x, mode='eval')
    pred_ids = trim_pred_ids(logits_to_ids(logits[0]), float(pred_len[0].item()))
    true_ids = normalize_ids(x[0])
    true_selfies = ids_to_selfies(true_ids)
    pred_selfies = ids_to_selfies(pred_ids)
    true_smiles = selfies_to_smiles(true_selfies)
    pred_smiles = selfies_to_smiles(pred_selfies)
    rows.append({
        'row_index': i,
        'true_selfies': true_selfies,
        'pred_selfies': pred_selfies,
        'true_smiles': true_smiles,
        'pred_smiles': pred_smiles,
        'canonical_match': canonical_smiles(true_smiles) == canonical_smiles(pred_smiles),
    })

reconstruction_examples = pd.DataFrame(rows)
reconstruction_csv = TABLES_DIR / 'reconstruction_spot_checks.csv'
reconstruction_examples.to_csv(reconstruction_csv, index=False)
print('Saved reconstruction spot checks to', reconstruction_csv)
display(reconstruction_examples)


## Latent PCA And t-SNE

These plots keep the same latent checks as the training notebook:

- PCA colored by molecule length
- PCA colored by the presence of oxygen
- t-SNE colored by the presence of oxygen

The only cleanup is that the encoding path and labeling are now consistent with the shared helpers.

In [ ]:
n_samples = min(5000, len(data))
vis_data = data[:n_samples]
vis_loader = DataLoader(torch.as_tensor(vis_data, dtype=torch.long), batch_size=256, shuffle=False)
zs = []
with torch.no_grad():
    for x in tqdm(vis_loader, desc='Encoding latent subset'):
        x = x.to(device)
        mu, _ = model.encode(x)
        zs.append(mu.detach().cpu().numpy())

z = np.concatenate(zs, axis=0)
sample_size = min(5000, z.shape[0])
idxs = np.random.choice(z.shape[0], size=sample_size, replace=False)
z_sample = z[idxs]
len_labels = np.array(df['lenghts'])[:n_samples][idxs]
oxygen_labels = df['selfies'].apply(lambda x: 1 if 'O' in x else 0).to_numpy()[:n_samples][idxs]

pca_z_2d = PCA(n_components=2, random_state=SEED).fit_transform(z_sample)
projection_df = pd.DataFrame({
    'dataset_idx': idxs,
    'length': len_labels,
    'has_oxygen': oxygen_labels,
    'pca1': pca_z_2d[:, 0],
    'pca2': pca_z_2d[:, 1],
    'tsne1': np.nan,
    'tsne2': np.nan,
})

plt.figure(figsize=(11, 8))
plt.scatter(pca_z_2d[:, 0], pca_z_2d[:, 1], c=len_labels, cmap='viridis', alpha=0.7, s=16)
plt.title('z colored by molecule length (PCA)')
plt.colorbar(label='length')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
pca_len_path = PLOTS_DIR / 'latent_pca_length.png'
plt.savefig(pca_len_path, dpi=160)
plt.show()

plt.figure(figsize=(11, 8))
for cls in np.unique(oxygen_labels)[::-1]:
    mask = oxygen_labels == cls
    plt.scatter(pca_z_2d[mask, 0], pca_z_2d[mask, 1], label=f'class {cls}', alpha=0.7, s=16)
plt.title('Has oxygen or not (PCA)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
pca_oxygen_path = PLOTS_DIR / 'latent_pca_oxygen.png'
plt.savefig(pca_oxygen_path, dpi=160)
plt.show()

tsne_size = min(2000, len(z_sample))
tsne_idxs = np.random.choice(len(z_sample), size=tsne_size, replace=False)
tsne_z = TSNE(n_components=2, random_state=SEED, init='pca', learning_rate='auto').fit_transform(z_sample[tsne_idxs])
projection_df.loc[tsne_idxs, 'tsne1'] = tsne_z[:, 0]
projection_df.loc[tsne_idxs, 'tsne2'] = tsne_z[:, 1]

plt.figure(figsize=(11, 8))
for cls in np.unique(oxygen_labels)[::-1]:
    mask = oxygen_labels[tsne_idxs] == cls
    plt.scatter(tsne_z[mask, 0], tsne_z[mask, 1], label=f'class {cls}', alpha=0.7, s=16)
plt.title('Has oxygen or not (t-SNE)')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.legend()
plt.tight_layout()
tsne_path = PLOTS_DIR / 'latent_tsne_oxygen.png'
plt.savefig(tsne_path, dpi=160)
plt.show()

projection_csv = TABLES_DIR / 'latent_projection_sample.csv'
projection_df.to_csv(projection_csv, index=False)
print('Saved PCA length plot to', pca_len_path)
print('Saved PCA oxygen plot to', pca_oxygen_path)
print('Saved t-SNE plot to', tsne_path)
print('Saved projection table to', projection_csv)


## Random Latent Sampling Validity

The training notebook sampled random latent vectors and decoded them back into molecules. This cell keeps that test and reports a validity ratio with a sample of decoded molecules.

In [ ]:
@torch.no_grad()
def sample_and_validate(model, n_samples, latent_scale=1.0):
    z = latent_scale * torch.randn(n_samples, latent_size, device=device)
    decoded_df = decode_latent_batch(z)
    valid_ratio = float(decoded_df['is_valid'].mean())
    return decoded_df, valid_ratio

sampled_df, valid_ratio = sample_and_validate(model, n_samples=100, latent_scale=1.0)
sampled_csv = TABLES_DIR / 'random_latent_sampling.csv'
sampled_df.to_csv(sampled_csv, index=False)
print('Validity:', valid_ratio)
print('Saved sampled latent decodes to', sampled_csv)
display(sampled_df[['is_valid', 'smiles', 'selfies']].head(25))


## Latent Traversals

These are the same three traversal styles used after training:

- move along one latent dimension
- take a random walk in latent space
- linearly interpolate between two encoded molecules

The helpers below return data frames instead of relying on ad-hoc printing, but the tests themselves are the same.

In [ ]:
def latent_traversal(model, x_row, dim=0, steps=40, max_delta=20.0):
    x_tensor = tensorize_rows(x_row).unsqueeze(0)
    with torch.no_grad():
        mu, _ = model.encode(x_tensor)
    z0 = mu[0].detach().cpu().numpy()
    deltas = np.linspace(-max_delta, max_delta, steps, dtype=np.float32)
    z_batch = np.repeat(z0[None, :], len(deltas), axis=0)
    z_batch[:, dim] += deltas
    decoded_df = decode_latent_batch(z_batch)
    decoded_df.insert(0, 'delta', deltas)
    return decoded_df

def latent_random_traversal(model, data_array, steps=30, sigma=0.1):
    x = tensorize_rows(data_array[np.random.randint(len(data_array))]).unsqueeze(0)
    with torch.no_grad():
        mu, _ = model.encode(x)
    z = mu.clone()
    rows = []
    for step in range(steps):
        z = z + sigma * torch.randn_like(z)
        decoded_df = decode_latent_batch(z)
        record = decoded_df.iloc[0].to_dict()
        record['step'] = step
        rows.append(record)
    walk_df = pd.DataFrame(rows)
    labels = walk_df['smiles'].fillna('<INVALID>').tolist()
    unique, counts = count_duplicates(labels)
    segment_df = pd.DataFrame({'smiles': unique, 'steps': counts})
    return walk_df, segment_df

def latent_linear_traversal(model, data_array, steps=40):
    idx1 = np.random.randint(len(data_array))
    idx2 = np.random.randint(len(data_array))
    x1 = tensorize_rows(data_array[idx1]).unsqueeze(0)
    x2 = tensorize_rows(data_array[idx2]).unsqueeze(0)
    with torch.no_grad():
        z1, _ = model.encode(x1)
        z2, _ = model.encode(x2)
    alphas = np.linspace(0.0, 1.0, steps, dtype=np.float32)
    z_batch = np.vstack([(1.0 - a) * z1[0].detach().cpu().numpy() + a * z2[0].detach().cpu().numpy() for a in alphas])
    decoded_df = decode_latent_batch(z_batch)
    decoded_df.insert(0, 'alpha', alphas)
    decoded_df.attrs['start_idx'] = int(idx1)
    decoded_df.attrs['end_idx'] = int(idx2)
    return decoded_df

one_d_df = latent_traversal(model, train_data[np.random.randint(0, len(train_data))], dim=1, steps=40, max_delta=20.0)
one_d_csv = TABLES_DIR / 'latent_traversal_1d.csv'
one_d_df.to_csv(one_d_csv, index=False)
display(one_d_df[['delta', 'smiles', 'is_valid']])

random_walk_df, random_walk_segments = latent_random_traversal(model, val_data, steps=30, sigma=0.1)
random_walk_csv = TABLES_DIR / 'latent_random_walk.csv'
random_walk_segments_csv = TABLES_DIR / 'latent_random_walk_segments.csv'
random_walk_df.to_csv(random_walk_csv, index=False)
random_walk_segments.to_csv(random_walk_segments_csv, index=False)
print('Random walk segments:')
display(random_walk_segments)

linear_interp_df = latent_linear_traversal(model, val_data, steps=40)
linear_interp_df.insert(1, 'start_idx', linear_interp_df.attrs['start_idx'])
linear_interp_df.insert(2, 'end_idx', linear_interp_df.attrs['end_idx'])
linear_interp_csv = TABLES_DIR / 'latent_linear_interpolation.csv'
linear_interp_df.to_csv(linear_interp_csv, index=False)
print('Interpolation endpoints:', linear_interp_df.attrs['start_idx'], '->', linear_interp_df.attrs['end_idx'])
display(linear_interp_df[['alpha', 'smiles', 'is_valid']])

print('Saved 1D traversal to', one_d_csv)
print('Saved random walk to', random_walk_csv)
print('Saved random walk segments to', random_walk_segments_csv)
print('Saved linear interpolation to', linear_interp_csv)


## Token Position Probe From Latent

This keeps the training notebook's quick probe: fit a ridge regressor from the latent mean to one token position and report the accuracy after rounding back to token IDs.

In [ ]:
pos_to_probe = 17
latents = []
targets = []

for x in tqdm(val_data, desc='Building probe set'):
    if pos_to_probe >= len(x):
        continue
    x_tensor = tensorize_rows(x).unsqueeze(0)
    with torch.no_grad():
        mu, _ = model.encode(x_tensor)
    latents.append(mu.squeeze(0).detach().cpu().numpy())
    targets.append(int(x[pos_to_probe]))

latents = np.stack(latents)
targets = np.array(targets)

X_train, X_test, y_train, y_test = train_test_split(latents, targets, test_size=0.2, random_state=42)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred = ridge.predict(X_test)
y_pred_ids = np.clip(np.round(y_pred).astype(int), 0, len(id2tok) - 1)
acc = accuracy_score(y_test, y_pred_ids)
token_probe_df = pd.DataFrame([
    {'token_position': pos_to_probe, 'accuracy': float(acc), 'n_train': len(X_train), 'n_test': len(X_test)}
])
token_probe_csv = TABLES_DIR / 'token_position_probe.csv'
token_probe_df.to_csv(token_probe_csv, index=False)
print(f'Prediction accuracy for token at position {pos_to_probe}: {acc:.4f}')
print('Saved token probe metrics to', token_probe_csv)


## 3D PCA View

This is the same 3D PCA visualization used after training, but it renders inline in the notebook instead of forcing a browser renderer.

In [ ]:
pca_z_3d = PCA(n_components=3, random_state=SEED).fit_transform(z_sample)
fig = px.scatter_3d(
    x=pca_z_3d[:, 0],
    y=pca_z_3d[:, 1],
    z=pca_z_3d[:, 2],
    color=len_labels,
    labels={'color': 'length'},
    title='Latent space PCA (3D)',
)
pca3_html = PLOTS_DIR / 'latent_pca_3d.html'
fig.write_html(str(pca3_html), include_plotlyjs='cdn')
print('Saved 3D PCA html to', pca3_html)
fig.show()


## Local Latent Grid
This test checks **local smoothness** around one encoded validation molecule. The seed molecule sits at the center of a 2D grid built from the first two PCA directions of the validation latent cloud.

Interpretation:
- Center cells should stay close to the seed and show whether the decoder is stable in its immediate neighborhood.
- Repeated molecules across nearby cells usually mean the latent space is locally flat or invariant in that region.
- More distinct but still valid molecules indicate controlled semantic drift away from the seed.
- Invalid molecules near the edges usually mean the traversal has left the locally stable part of the manifold.
- The saved fingerprint distance to the seed is a coarse measure of how much each decoded molecule drifts semantically.


In [ ]:

def mol_distance(mol1, mol2, radius=2, nBits=2048):
    if mol1 is None or mol2 is None:
        return None
    fp1 = make_morgan_fp(mol1, radius=radius, nBits=nBits)
    fp2 = make_morgan_fp(mol2, radius=radius, nBits=nBits)
    return 1.0 - DataStructs.TanimotoSimilarity(fp1, fp2)

def latent_grid_unique_plot(model, x_row, direction_x, direction_y, grid_size=7, delta=7.0, noise=0.0, mol_size=(200, 200)):
    x_tensor = tensorize_rows(x_row)
    true_smiles = ids_to_smiles(x_tensor)
    true_mol = mol_from_smiles(true_smiles)
    print('True molecule from dataset:', true_smiles)
    if true_mol is not None:
        display(Draw.MolToImage(true_mol, size=(300, 300)))

    with torch.no_grad():
        mu, _ = model.encode(x_tensor.unsqueeze(0))
    z0 = mu[0]

    seed_df = decode_latent_batch(z0)
    seed_smiles = seed_df.iloc[0]['smiles']
    seed_canonical = seed_df.iloc[0]['canonical_smiles']
    seed_mol = mol_from_smiles(seed_smiles)
    print('Decoded seed molecule:', seed_smiles)
    if seed_mol is not None:
        display(Draw.MolToImage(seed_mol, size=(300, 300)))

    direction_x = torch.as_tensor(direction_x, dtype=torch.float32, device=device)
    direction_y = torch.as_tensor(direction_y, dtype=torch.float32, device=device)
    direction_x = direction_x / (direction_x.norm() + 1e-8)
    direction_y = direction_y / (direction_y.norm() + 1e-8)
    offsets = torch.linspace(-delta, delta, grid_size, device=device)

    mols = []
    legends = []
    seen = {}
    records = []

    for row_idx, dy in enumerate(offsets.detach().cpu().tolist()):
        for col_idx, dx in enumerate(offsets.detach().cpu().tolist()):
            z = z0.clone() + float(dx) * direction_x + float(dy) * direction_y
            if noise > 0:
                z = z + noise * torch.randn_like(z)
            decoded_df = decode_latent_batch(z)
            decoded = decoded_df.iloc[0]
            smi = decoded['smiles']
            canon = decoded['canonical_smiles']
            valid = int(decoded['is_valid'])
            mol = mol_from_smiles(smi) if smi is not None else None
            if smi is None:
                mols.append(None)
                legends.append('')
            else:
                if smi not in seen:
                    seen[smi] = mol
                mols.append(seen[smi])
                if seed_mol is not None and mol is not None:
                    legends.append(f'd={mol_distance(seed_mol, mol):.2f}')
                else:
                    legends.append('')

            distance_to_seed = mol_distance(seed_mol, mol) if seed_mol is not None and mol is not None else np.nan
            records.append({
                'grid_row': int(row_idx),
                'grid_col': int(col_idx),
                'offset_x': float(dx),
                'offset_y': float(dy),
                'seed_smiles': seed_smiles,
                'seed_canonical_smiles': seed_canonical,
                'decoded_smiles': smi,
                'canonical_smiles': canon,
                'is_valid': valid,
                'distance_to_seed': distance_to_seed,
            })

    center = (grid_size * grid_size) // 2
    legends[center] = 'SEED\n' + legends[center]
    display_img = Draw.MolsToGridImage(mols, molsPerRow=grid_size, subImgSize=mol_size, legends=legends)
    save_img = Draw.MolsToGridImage(mols, molsPerRow=grid_size, subImgSize=mol_size, legends=legends, useSVG=False, returnPNG=True)
    display(display_img)

    grid_table = pd.DataFrame(records)
    grid_png = PLOTS_DIR / 'local_latent_grid.png'
    grid_csv = TABLES_DIR / 'local_latent_grid_cells.csv'
    save_rdkit_image(save_img, grid_png)
    grid_table.to_csv(grid_csv, index=False)
    print('Saved local latent grid image to', grid_png)
    print('Saved local latent grid table to', grid_csv)
    return grid_table

val_latents = encode_numpy_rows(val_data)
principal_pca = PCA(n_components=8, random_state=SEED)
principal_pca.fit(val_latents)

x = val_data[np.random.randint(len(val_data))]
local_latent_grid_df = latent_grid_unique_plot(
    model,
    x,
    direction_x=principal_pca.components_[0],
    direction_y=principal_pca.components_[1],
    grid_size=7,
    delta=7.0,
    noise=0.0,
)


## Grouped Chemistry Semantics
This section checks whether simple chemistry families organize coherently in latent space.

Interpretation:
- The PCA plot asks whether related families, such as alkanes and alcohols, cluster in a structured way instead of scattering randomly.
- Ordered homologous-series distances test whether adding one carbon at a time produces a reasonably consistent latent step size.
- Comparing alcohol-minus-alkane vectors checks whether the latent space reuses a similar direction for the same chemical transformation across chain lengths.
- High cosine similarity for the OH direction suggests the model has learned a reusable semantic direction instead of encoding every example independently.


In [ ]:
molecule_groups = {
    'alkanes': ['C', 'CC', 'CCC', 'CCCC'],
    'alcohols': ['CO', 'CCO', 'CCCO', 'CCCCO'],
}

def encode_groups(model, molecule_groups, tok2id, max_len, device):
    model.eval()
    Z = []
    labels = []
    names = []
    with torch.no_grad():
        for group, smiles_list in molecule_groups.items():
            for smi in smiles_list:
                x = smiles_to_ids(smi, tok2id, max_len)
                x = torch.as_tensor(x, dtype=torch.long, device=device).unsqueeze(0)
                mu, _ = model.encode(x)
                Z.append(mu.squeeze(0).detach().cpu().numpy())
                labels.append(group)
                names.append(smi)
    return np.vstack(Z), labels, names

def plot_group_pca(Z, labels, names):
    pca = PCA(n_components=2, random_state=SEED)
    Z2 = pca.fit_transform(Z)
    plt.figure(figsize=(8, 6))
    for group in sorted(set(labels)):
        idx = [i for i, g in enumerate(labels) if g == group]
        plt.scatter(Z2[idx, 0], Z2[idx, 1], label=group, s=80)
    for i, name in enumerate(names):
        plt.text(Z2[i, 0], Z2[i, 1], name, fontsize=9)
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.title('Latent space PCA - chemical semantics')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    group_pca_path = PLOTS_DIR / 'grouped_chemistry_pca.png'
    plt.savefig(group_pca_path, dpi=160)
    plt.show()
    return Z2, group_pca_path

Z, labels, names = encode_groups(model, molecule_groups, tok2id, max_len, device)
Z2, group_pca_path = plot_group_pca(Z, labels, names)

group_meta_df = pd.DataFrame({
    'group': labels,
    'smiles': names,
    'pca1': Z2[:, 0],
    'pca2': Z2[:, 1],
})
latent_dim_df = pd.DataFrame(
    Z,
    columns=[f'z_{dim:03d}' for dim in range(Z.shape[1])],
)
group_coord_df = pd.concat([group_meta_df, latent_dim_df], axis=1, copy=False)

group_coords_csv = TABLES_DIR / 'grouped_chemistry_coordinates.csv'
group_coord_df.to_csv(group_coords_csv, index=False)
print('Saved grouped chemistry PCA to', group_pca_path)
print('Saved grouped chemistry coordinates to', group_coords_csv)


In [ ]:
def latent_distances(model, smiles_list, tok2id, max_len, device):
    zs = []
    with torch.no_grad():
        for smi in smiles_list:
            x = smiles_to_ids(smi, tok2id, max_len)
            x = torch.as_tensor(x, dtype=torch.long, device=device).unsqueeze(0)
            mu, _ = model.encode(x)
            zs.append(mu.squeeze(0).detach().cpu().numpy())
    return [float(np.linalg.norm(zs[i + 1] - zs[i])) for i in range(len(zs) - 1)]

def latent_vector(model, s1, s2, tok2id, max_len, device):
    z1 = encode_smiles(s1)
    z2 = encode_smiles(s2)
    return z1 - z2

alkanes = ['C', 'CC', 'CCC', 'CCCC', 'CCCCC', 'CCCCCC', 'CCCCCCC']
alcohols = ['CO', 'CCO', 'CCCO', 'CCCCO', 'CCCCCO', 'CCCCCCO', 'CCCCCCCO']
amines = ['CN', 'CCN', 'CCCN', 'CCCCN', 'CCCCCN', 'CCCCCCN', 'CCCCCCCN']
acids = ['C(=O)O', 'CC(=O)O', 'CCC(=O)O', 'CCCC(=O)O', 'CCCCC(=O)O', 'CCCCCC(=O)O', 'CCCCCCC(=O)O']

series_lookup = {
    'alkanes': alkanes,
    'alcohols': alcohols,
    'amines': amines,
    'acids': acids,
}

for series_name, smiles_list in series_lookup.items():
    dists = latent_distances(model, smiles_list, tok2id, max_len, device)
    print(f'{series_name.title()} latent distances:', dists)
    series_df = pd.DataFrame({
        'series': series_name,
        'start_smiles': smiles_list[:-1],
        'end_smiles': smiles_list[1:],
        'latent_distance': dists,
    })
    series_csv = TABLES_DIR / f'grouped_semantics_{series_name}_distances.csv'
    series_df.to_csv(series_csv, index=False)
    print('Saved', series_name, 'distance table to', series_csv)

v1 = latent_vector(model, 'CCCO', 'CCC', tok2id, max_len, device)
v2 = latent_vector(model, 'CCCCO', 'CCCC', tok2id, max_len, device)
cos_sim = torch.nn.functional.cosine_similarity(v1, v2, dim=0)
oh_direction_df = pd.DataFrame([
    {
        'transform_a': 'CCCO-CCC',
        'transform_b': 'CCCCO-CCCC',
        'cosine_similarity': float(cos_sim.item()),
    }
])
oh_direction_csv = TABLES_DIR / 'oh_direction_cosine.csv'
oh_direction_df.to_csv(oh_direction_csv, index=False)
print('Cosine similarity (OH direction):', float(cos_sim.item()))
print('Saved OH direction cosine result to', oh_direction_csv)



## Saved Artifact Summary
All generated plots, tables, and notes are written under artifacts/latent_test_analysis. The notebook keeps the analysis inline, but every major section now also leaves a reproducible artifact trail on disk for later comparison.
